# Day 4: Deep Learning Models — Vietnamese Price Prediction

**6 model families, ~12 experiments:**
- Model 0: DNN ResidualBlock (HashingVec / TF-IDF, hidden=2048→4096)
- Model 1: PhoBERT-base-v2 fine-tune (regression head)
- Model 2: Vietnamese Embedding + MLP (dangvantuan / AITeamVN)
- Model 3: XLM-RoBERTa-base fine-tune (multilingual)
- Model 4: Vietnamese Embedding + LightGBM (hybrid DL+ML)
- Model 5: Vietnamese Embedding + DNN ResidualBlock

**Dataset:** SeanSunny/items_tv_v6 filtered <= 1M VND
**Baseline (Day 3):** Blended RMSLE=0.5164


In [ ]:
# Imports

import sys
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.sparse import hstack

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from pricer_vi.items import Item
from pricer_vi.deep_neural_network import (
    DeepNeuralNetwork, MLP, train_torch_model, predict_batch,
)


In [ ]:
# Config

DAY4 = Path(__file__).resolve().parent
DATASET = "SeanSunny/items_tv_v6"
MAX_PRICE = 1_000_000
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

ALL_RESULTS = {}


In [ ]:
# Helper: metrics

def rmsle(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_batch(y_true, y_pred, name="Model"):
    """Evaluate predictions, print metrics, store in ALL_RESULTS."""
    r = {
        "rmsle": rmsle(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape": mape(y_true, y_pred),
        "r2": r2_score(y_true, y_pred) * 100,
    }
    print(f"\n{'='*60}")
    print(f"{name} ({len(y_true)} items)")
    print(f"  RMSLE:  {r['rmsle']:.4f}")
    print(f"  MAE:    {r['mae']:,.0f} VND")
    print(f"  MAPE:   {r['mape']:.1f}%")
    print(f"  R2:     {r['r2']:.1f}%")
    print(f"{'='*60}")
    ALL_RESULTS[name] = r
    return r


# ============================================================
# PHASE 1: LOAD DATA + TOKENIZE + EMBED + VECTORIZE (CACHE)
# ============================================================


In [ ]:
# Phase 1a: Load data from HuggingFace

print("\n--- Phase 1a: Loading data ---")
train_items, val_items, test_items = Item.from_hub(DATASET)

def filter_items(items, max_price=MAX_PRICE):
    return [it for it in items if 0 < it.price <= max_price]

train_items = filter_items(train_items)
val_items = filter_items(val_items)
test_items = filter_items(test_items)

print(f"Train: {len(train_items):,} | Val: {len(val_items):,} | Test: {len(test_items):,}")

train_summaries = [it.summary for it in train_items]
val_summaries = [it.summary for it in val_items]
test_summaries = [it.summary for it in test_items]

train_prices = np.array([it.price for it in train_items], dtype=np.float32)
val_prices = np.array([it.price for it in val_items], dtype=np.float32)
test_prices = np.array([it.price for it in test_items], dtype=np.float32)

train_categories = [it.category for it in train_items]
val_categories = [it.category for it in val_items]
test_categories = [it.category for it in test_items]


In [ ]:
# Phase 1b: Category one-hot encoding

print("\n--- Phase 1b: Category one-hot ---")
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
cat_train = cat_encoder.fit_transform(np.array(train_categories).reshape(-1, 1))
cat_val = cat_encoder.transform(np.array(val_categories).reshape(-1, 1))
cat_test = cat_encoder.transform(np.array(test_categories).reshape(-1, 1))
print(f"Categories: {cat_encoder.categories_[0].tolist()}")
print(f"One-hot features: {cat_train.shape[1]}")


In [ ]:
# Phase 1c: Underthesea tokenize + cache

UNDERTHESEA_CACHE = {
    "train": DAY4 / "tokenized_train_1m.pkl",
    "val": DAY4 / "tokenized_val_1m.pkl",
    "test": DAY4 / "tokenized_test_1m.pkl",
}

# Also check Day 3 cache
DAY3 = DAY4.parent / "day3"

def load_or_tokenize_underthesea(summaries, cache_path, label=""):
    if cache_path.exists():
        print(f"  Loading underthesea cache: {cache_path.name}")
        return joblib.load(cache_path)
    # Check Day 3 cache
    day3_path = DAY3 / cache_path.name
    if day3_path.exists():
        print(f"  Loading Day 3 cache: {day3_path}")
        data = joblib.load(day3_path)
        joblib.dump(data, cache_path)
        return data
    print(f"  Tokenizing {label} ({len(summaries):,} docs) with underthesea...")
    from underthesea import word_tokenize
    tokenized = [word_tokenize(s, format="text") for s in summaries]
    joblib.dump(tokenized, cache_path)
    print(f"  Saved: {cache_path.name}")
    return tokenized

print("\n--- Phase 1c: Underthesea tokenize ---")
tok_train = load_or_tokenize_underthesea(train_summaries, UNDERTHESEA_CACHE["train"], "train")
tok_val = load_or_tokenize_underthesea(val_summaries, UNDERTHESEA_CACHE["val"], "val")
tok_test = load_or_tokenize_underthesea(test_summaries, UNDERTHESEA_CACHE["test"], "test")


In [ ]:
# Phase 1d: pyvi tokenize + cache (for dangvantuan embedding)

PYVI_CACHE = {
    "train": DAY4 / "pyvi_tokenized_train.pkl",
    "val": DAY4 / "pyvi_tokenized_val.pkl",
    "test": DAY4 / "pyvi_tokenized_test.pkl",
}

def load_or_tokenize_pyvi(summaries, cache_path, label=""):
    if cache_path.exists():
        print(f"  Loading pyvi cache: {cache_path.name}")
        return joblib.load(cache_path)
    print(f"  Tokenizing {label} ({len(summaries):,} docs) with pyvi...")
    from pyvi.ViTokenizer import tokenize
    tokenized = [tokenize(s) for s in summaries]
    joblib.dump(tokenized, cache_path)
    print(f"  Saved: {cache_path.name}")
    return tokenized

print("\n--- Phase 1d: pyvi tokenize ---")
pyvi_train = load_or_tokenize_pyvi(train_summaries, PYVI_CACHE["train"], "train")
pyvi_val = load_or_tokenize_pyvi(val_summaries, PYVI_CACHE["val"], "val")
pyvi_test = load_or_tokenize_pyvi(test_summaries, PYVI_CACHE["test"], "test")


In [ ]:
# Phase 1e: dangvantuan embedding extraction + cache

EMB_DV = {
    "train": DAY4 / "dangvantuan_train.npy",
    "val": DAY4 / "dangvantuan_val.npy",
    "test": DAY4 / "dangvantuan_test.npy",
}

def load_or_embed_dangvantuan(texts_dict, cache_dict):
    """texts_dict: {split: [pyvi_tokenized_texts]}, cache_dict: {split: Path}"""
    all_loaded = all(p.exists() for p in cache_dict.values())
    if all_loaded:
        print("  Loading dangvantuan cache...")
        return {k: np.load(v) for k, v in cache_dict.items()}

    print("  Loading dangvantuan/vietnamese-embedding model...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("dangvantuan/vietnamese-embedding")

    result = {}
    for split, texts in texts_dict.items():
        path = cache_dict[split]
        if path.exists():
            print(f"  Loading cache: {path.name}")
            result[split] = np.load(path)
        else:
            print(f"  Encoding {split} ({len(texts):,} docs)...")
            emb = model.encode(texts, batch_size=128, show_progress_bar=True,
                               normalize_embeddings=True)
            np.save(path, emb.astype(np.float32))
            print(f"  Saved: {path.name} shape={emb.shape}")
            result[split] = emb
    del model
    torch.cuda.empty_cache()
    return result

print("\n--- Phase 1e: dangvantuan embeddings ---")
dv_emb = load_or_embed_dangvantuan(
    {"train": pyvi_train, "val": pyvi_val, "test": pyvi_test}, EMB_DV
)


In [ ]:
# Phase 1f: AITeamVN embedding extraction + cache

EMB_AT = {
    "train": DAY4 / "aiteamvn_train.npy",
    "val": DAY4 / "aiteamvn_val.npy",
    "test": DAY4 / "aiteamvn_test.npy",
}

def load_or_embed_aiteamvn(texts_dict, cache_dict):
    """texts_dict: {split: [raw_texts]}, cache_dict: {split: Path}"""
    all_loaded = all(p.exists() for p in cache_dict.values())
    if all_loaded:
        print("  Loading AITeamVN cache...")
        return {k: np.load(v) for k, v in cache_dict.items()}

    print("  Loading AITeamVN/Vietnamese_Embedding model...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("AITeamVN/Vietnamese_Embedding")
    model.max_seq_length = 2048

    result = {}
    for split, texts in texts_dict.items():
        path = cache_dict[split]
        if path.exists():
            print(f"  Loading cache: {path.name}")
            result[split] = np.load(path)
        else:
            print(f"  Encoding {split} ({len(texts):,} docs)...")
            emb = model.encode(texts, batch_size=64, show_progress_bar=True)
            np.save(path, emb.astype(np.float32))
            print(f"  Saved: {path.name} shape={emb.shape}")
            result[split] = emb
    del model
    torch.cuda.empty_cache()
    return result

print("\n--- Phase 1f: AITeamVN embeddings ---")
at_emb = load_or_embed_aiteamvn(
    {"train": train_summaries, "val": val_summaries, "test": test_summaries}, EMB_AT
)


In [ ]:
# Phase 1g: HashingVectorizer + cache

HV_CACHE = DAY4 / "hashing_vectorizer.pkl"

print("\n--- Phase 1g: HashingVectorizer ---")
if HV_CACHE.exists():
    hv = joblib.load(HV_CACHE)
    print("  Loaded HashingVectorizer cache")
else:
    hv = HashingVectorizer(n_features=5000, binary=True, alternate_sign=False)
    joblib.dump(hv, HV_CACHE)
    print("  Created HashingVectorizer (stateless)")

X_hv_train = hv.transform(train_summaries)
X_hv_val = hv.transform(val_summaries)
X_hv_test = hv.transform(test_summaries)
print(f"  HashingVec shape: {X_hv_train.shape}")


In [ ]:
# Phase 1h: TF-IDF Vectorizer + cache

TFIDF_CACHE = DAY4 / "tfidf_vectorizer.pkl"

print("\n--- Phase 1h: TF-IDF Vectorizer ---")
if TFIDF_CACHE.exists():
    tfidf = joblib.load(TFIDF_CACHE)
    print("  Loaded TF-IDF cache")
    X_tfidf_train = tfidf.transform(tok_train)
else:
    tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), sublinear_tf=True)
    X_tfidf_train = tfidf.fit_transform(tok_train)
    joblib.dump(tfidf, TFIDF_CACHE)
    print("  Fitted TF-IDF and saved cache")

X_tfidf_val = tfidf.transform(tok_val)
X_tfidf_test = tfidf.transform(tok_test)
print(f"  TF-IDF shape: {X_tfidf_train.shape}")


In [ ]:
# Phase 1 complete

print("\n" + "=" * 60)
print("PHASE 1 COMPLETE — All data loaded and cached")
print(f"  Underthesea tokenized: {len(tok_train):,} train")
print(f"  pyvi tokenized: {len(pyvi_train):,} train")
print(f"  dangvantuan embeddings: {dv_emb['train'].shape}")
print(f"  AITeamVN embeddings: {at_emb['train'].shape}")
print(f"  HashingVec: {X_hv_train.shape}")
print(f"  TF-IDF: {X_tfidf_train.shape}")
print("=" * 60)


# ============================================================
# PHASE 2: MODEL 0 — DNN ResidualBlock (bag-of-words)
# ============================================================


In [ ]:
# Helper: prepare DNN data

def prepare_dnn_data(X_sparse, cat_sparse, prices, add_cat=True):
    """Convert sparse features + category to dense torch tensor."""
    if add_cat:
        X = hstack([X_sparse, cat_sparse]).toarray()
    else:
        X = X_sparse.toarray() if hasattr(X_sparse, 'toarray') else X_sparse
    return (
        torch.FloatTensor(X),
        torch.FloatTensor(prices).unsqueeze(1),
    )


In [ ]:
# Phase 2: Model 0a — DNN + HashingVec (hidden=2048)

print("\n\n" + "#" * 60)
print("# MODEL 0a: DNN + HashingVec (hidden=2048)")
print("#" * 60)

X_tr, y_tr = prepare_dnn_data(X_hv_train, cat_train, train_prices)
X_va, y_va = prepare_dnn_data(X_hv_val, cat_val, val_prices)
X_te, y_te = prepare_dnn_data(X_hv_test, cat_test, test_prices)

model_0a = DeepNeuralNetwork(X_tr.shape[1], hidden_size=2048, num_layers=10)
model_0a, y_mean_0a, y_std_0a, hist_0a = train_torch_model(
    model_0a, X_tr, y_tr, X_va, y_va, DEVICE, epochs=5, batch_size=64
)
pred_0a = predict_batch(model_0a, X_te, y_mean_0a, y_std_0a, DEVICE)
evaluate_batch(test_prices, pred_0a, "Model 0a: DNN+HashingVec (h=2048)")

del model_0a, X_tr, X_va, X_te
torch.cuda.empty_cache()


In [ ]:
# Phase 2: Model 0b — DNN + TF-IDF (hidden=2048)

print("\n\n" + "#" * 60)
print("# MODEL 0b: DNN + TF-IDF (hidden=2048)")
print("#" * 60)

X_tr, y_tr = prepare_dnn_data(X_tfidf_train, cat_train, train_prices)
X_va, y_va = prepare_dnn_data(X_tfidf_val, cat_val, val_prices)
X_te, y_te = prepare_dnn_data(X_tfidf_test, cat_test, test_prices)

model_0b = DeepNeuralNetwork(X_tr.shape[1], hidden_size=2048, num_layers=10)
model_0b, y_mean_0b, y_std_0b, hist_0b = train_torch_model(
    model_0b, X_tr, y_tr, X_va, y_va, DEVICE, epochs=5, batch_size=64
)
pred_0b = predict_batch(model_0b, X_te, y_mean_0b, y_std_0b, DEVICE)
evaluate_batch(test_prices, pred_0b, "Model 0b: DNN+TF-IDF (h=2048)")

del model_0b
torch.cuda.empty_cache()


In [ ]:
# Phase 2: Model 0c — DNN best vectorizer (hidden=4096)

print("\n\n" + "#" * 60)
print("# MODEL 0c: DNN best vectorizer (hidden=4096)")
print("#" * 60)
print("Using TF-IDF (same data as 0b, reuse X_tr/X_va/X_te)")

model_0c = DeepNeuralNetwork(X_tr.shape[1], hidden_size=4096, num_layers=10)
model_0c, y_mean_0c, y_std_0c, hist_0c = train_torch_model(
    model_0c, X_tr, y_tr, X_va, y_va, DEVICE, epochs=5, batch_size=64
)
pred_0c = predict_batch(model_0c, X_te, y_mean_0c, y_std_0c, DEVICE)
evaluate_batch(test_prices, pred_0c, "Model 0c: DNN+TF-IDF (h=4096)")

# Save best DNN model
best_0 = min(
    [("0a", ALL_RESULTS.get("Model 0a: DNN+HashingVec (h=2048)", {}).get("rmsle", 99)),
     ("0b", ALL_RESULTS.get("Model 0b: DNN+TF-IDF (h=2048)", {}).get("rmsle", 99)),
     ("0c", ALL_RESULTS.get("Model 0c: DNN+TF-IDF (h=4096)", {}).get("rmsle", 99))],
    key=lambda x: x[1]
)
print(f"\nBest Model 0 variant: {best_0[0]} (RMSLE={best_0[1]:.4f})")
torch.save(model_0c.state_dict(), DAY4 / "dnn_best.pth")
print(f"Saved: dnn_best.pth")

del model_0c, X_tr, X_va, X_te, y_tr, y_va, y_te
torch.cuda.empty_cache()


# ============================================================
# PHASE 3: EMBEDDING-BASED MODELS (MLP / LightGBM / DNN)
# ============================================================


In [ ]:
# Helper: prepare embedding data

def prepare_embed_data(emb_dict, cat_sparse, prices):
    """Combine embeddings + category one-hot → torch tensors."""
    cat_dense = cat_sparse.toarray().astype(np.float32)
    splits = {}
    for split in ("train", "val", "test"):
        X = np.hstack([emb_dict[split], cat_dense[:len(emb_dict[split])]])
        y = prices[split]
        splits[split] = (torch.FloatTensor(X), torch.FloatTensor(y).unsqueeze(1))
    return splits


prices_dict = {"train": train_prices, "val": val_prices, "test": test_prices}
cat_dict = {"train": cat_train, "val": cat_val, "test": cat_test}

def make_embed_tensors(emb_dict):
    """Combine embeddings + category → {split: (X_tensor, y_tensor)}."""
    result = {}
    for split in ("train", "val", "test"):
        cat_dense = cat_dict[split].toarray().astype(np.float32)
        X = np.hstack([emb_dict[split], cat_dense])
        y = prices_dict[split]
        result[split] = (torch.FloatTensor(X), torch.FloatTensor(y).unsqueeze(1))
    return result


In [ ]:
# Phase 3: Model 2a — dangvantuan embed + MLP

print("\n\n" + "#" * 60)
print("# MODEL 2a: dangvantuan embedding + MLP")
print("#" * 60)

dv_data = make_embed_tensors(dv_emb)
input_dim = dv_data["train"][0].shape[1]
print(f"Input dim: {input_dim} (768 embedding + {cat_train.shape[1]} category)")

model_2a = MLP(input_dim, hidden_sizes=(512, 256, 128))
model_2a, y_mean_2a, y_std_2a, _ = train_torch_model(
    model_2a, dv_data["train"][0], dv_data["train"][1],
    dv_data["val"][0], dv_data["val"][1], DEVICE,
    epochs=30, batch_size=128, lr=0.001, use_scheduler=False,
)
pred_2a = predict_batch(model_2a, dv_data["test"][0], y_mean_2a, y_std_2a, DEVICE)
evaluate_batch(test_prices, pred_2a, "Model 2a: dangvantuan+MLP")

del model_2a
torch.cuda.empty_cache()


In [ ]:
# Phase 3: Model 2b — AITeamVN embed + MLP

print("\n\n" + "#" * 60)
print("# MODEL 2b: AITeamVN embedding + MLP")
print("#" * 60)

at_data = make_embed_tensors(at_emb)
input_dim = at_data["train"][0].shape[1]
print(f"Input dim: {input_dim} (1024 embedding + {cat_train.shape[1]} category)")

model_2b = MLP(input_dim, hidden_sizes=(512, 256, 128))
model_2b, y_mean_2b, y_std_2b, _ = train_torch_model(
    model_2b, at_data["train"][0], at_data["train"][1],
    at_data["val"][0], at_data["val"][1], DEVICE,
    epochs=30, batch_size=128, lr=0.001, use_scheduler=False,
)
pred_2b = predict_batch(model_2b, at_data["test"][0], y_mean_2b, y_std_2b, DEVICE)
evaluate_batch(test_prices, pred_2b, "Model 2b: AITeamVN+MLP")

del model_2b
torch.cuda.empty_cache()


In [ ]:
# Phase 3: Model 4a — dangvantuan embed + LightGBM

print("\n\n" + "#" * 60)
print("# MODEL 4a: dangvantuan embedding + LightGBM")
print("#" * 60)

import lightgbm as lgb

dv_X_train = np.hstack([dv_emb["train"], cat_train.toarray()])
dv_X_val = np.hstack([dv_emb["val"], cat_val.toarray()])
dv_X_test = np.hstack([dv_emb["test"], cat_test.toarray()])
y_train_log = np.log1p(train_prices)
y_val_log = np.log1p(val_prices)

lgb_4a = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.1, num_leaves=31,
    n_jobs=6, random_state=SEED, verbose=-1,
)
lgb_4a.fit(
    dv_X_train, y_train_log,
    eval_set=[(dv_X_val, y_val_log)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)],
)
pred_4a = np.expm1(lgb_4a.predict(dv_X_test))
evaluate_batch(test_prices, pred_4a, "Model 4a: dangvantuan+LightGBM")


In [ ]:
# Phase 3: Model 4b — AITeamVN embed + LightGBM

print("\n\n" + "#" * 60)
print("# MODEL 4b: AITeamVN embedding + LightGBM")
print("#" * 60)

at_X_train = np.hstack([at_emb["train"], cat_train.toarray()])
at_X_val = np.hstack([at_emb["val"], cat_val.toarray()])
at_X_test = np.hstack([at_emb["test"], cat_test.toarray()])

lgb_4b = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.1, num_leaves=31,
    n_jobs=6, random_state=SEED, verbose=-1,
)
lgb_4b.fit(
    at_X_train, y_train_log,
    eval_set=[(at_X_val, y_val_log)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)],
)
pred_4b = np.expm1(lgb_4b.predict(at_X_test))
evaluate_batch(test_prices, pred_4b, "Model 4b: AITeamVN+LightGBM")


In [ ]:
# Phase 3: Model 5a — dangvantuan embed + DNN ResidualBlock

print("\n\n" + "#" * 60)
print("# MODEL 5a: dangvantuan embedding + DNN ResidualBlock")
print("#" * 60)

input_dim = dv_data["train"][0].shape[1]
model_5a = DeepNeuralNetwork(input_dim, hidden_size=1024, num_layers=6, dropout_prob=0.2)
model_5a, y_mean_5a, y_std_5a, _ = train_torch_model(
    model_5a, dv_data["train"][0], dv_data["train"][1],
    dv_data["val"][0], dv_data["val"][1], DEVICE,
    epochs=10, batch_size=64, lr=0.001,
)
pred_5a = predict_batch(model_5a, dv_data["test"][0], y_mean_5a, y_std_5a, DEVICE)
evaluate_batch(test_prices, pred_5a, "Model 5a: dangvantuan+DNN")

del model_5a
torch.cuda.empty_cache()


In [ ]:
# Phase 3: Model 5b — AITeamVN embed + DNN ResidualBlock

print("\n\n" + "#" * 60)
print("# MODEL 5b: AITeamVN embedding + DNN ResidualBlock")
print("#" * 60)

input_dim = at_data["train"][0].shape[1]
model_5b = DeepNeuralNetwork(input_dim, hidden_size=1024, num_layers=6, dropout_prob=0.2)
model_5b, y_mean_5b, y_std_5b, _ = train_torch_model(
    model_5b, at_data["train"][0], at_data["train"][1],
    at_data["val"][0], at_data["val"][1], DEVICE,
    epochs=10, batch_size=64, lr=0.001,
)
pred_5b = predict_batch(model_5b, at_data["test"][0], y_mean_5b, y_std_5b, DEVICE)
evaluate_batch(test_prices, pred_5b, "Model 5b: AITeamVN+DNN")

torch.save(model_5b.state_dict(), DAY4 / "embed_dnn_best.pth")
del model_5b
torch.cuda.empty_cache()


# ============================================================
# PHASE 4: MODEL 1 — PhoBERT-base-v2 fine-tune
# ============================================================


In [ ]:
# Phase 4: Model 1 — PhoBERT fine-tune

print("\n\n" + "#" * 60)
print("# MODEL 1: PhoBERT-base-v2 fine-tune (regression)")
print("#" * 60)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from torch.utils.data import Dataset as TorchDataset

PHOBERT_NAME = "vinai/phobert-base-v2"
PHOBERT_DIR = DAY4 / "phobert_best"

class PriceDataset(TorchDataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

print("Loading PhoBERT tokenizer...")
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME)

# PhoBERT expects underthesea-segmented text
print("Tokenizing with PhoBERT tokenizer...")
enc_train = phobert_tokenizer(tok_train, truncation=True, padding=True, max_length=256, return_tensors="pt")
enc_val = phobert_tokenizer(tok_val, truncation=True, padding=True, max_length=256, return_tensors="pt")
enc_test = phobert_tokenizer(tok_test, truncation=True, padding=True, max_length=256, return_tensors="pt")

y_train_log_t = torch.log1p(torch.FloatTensor(train_prices))
y_val_log_t = torch.log1p(torch.FloatTensor(val_prices))
y_test_log_t = torch.log1p(torch.FloatTensor(test_prices))

ds_train = PriceDataset(enc_train, y_train_log_t)
ds_val = PriceDataset(enc_val, y_val_log_t)
ds_test = PriceDataset(enc_test, y_test_log_t)

print("Loading PhoBERT model (num_labels=1 → regression)...")
phobert_model = AutoModelForSequenceClassification.from_pretrained(
    PHOBERT_NAME, num_labels=1, problem_type="regression"
)

training_args = TrainingArguments(
    output_dir=str(PHOBERT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=500,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=phobert_model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
)

print("Training PhoBERT...")
trainer.train()
trainer.save_model(str(PHOBERT_DIR))
phobert_tokenizer.save_pretrained(str(PHOBERT_DIR))
print(f"Saved: {PHOBERT_DIR}")

# Evaluate
print("Evaluating PhoBERT on test set...")
phobert_model.eval()
phobert_model.to(DEVICE)
pred_1_log = []
batch_size = 32
for i in range(0, len(test_items), batch_size):
    batch_enc = {k: v[i:i+batch_size].to(DEVICE) for k, v in enc_test.items()}
    with torch.no_grad():
        out = phobert_model(**batch_enc)
        pred_1_log.extend(out.logits.cpu().numpy().flatten())

pred_1 = np.expm1(np.array(pred_1_log))
pred_1 = np.clip(pred_1, 0, None)
evaluate_batch(test_prices, pred_1, "Model 1: PhoBERT-v2 fine-tune")

del phobert_model, trainer
torch.cuda.empty_cache()


# ============================================================
# PHASE 5: MODEL 3 — XLM-RoBERTa fine-tune
# ============================================================


In [ ]:
# Phase 5: Model 3 — XLM-RoBERTa fine-tune

print("\n\n" + "#" * 60)
print("# MODEL 3: XLM-RoBERTa-base fine-tune (regression)")
print("#" * 60)

XLMR_NAME = "FacebookAI/xlm-roberta-base"
XLMR_DIR = DAY4 / "xlmr_best"

print("Loading XLM-R tokenizer...")
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_NAME)

# XLM-R does NOT need Vietnamese word segmentation — use raw summaries
print("Tokenizing with XLM-R tokenizer...")
xlmr_enc_train = xlmr_tokenizer(train_summaries, truncation=True, padding=True, max_length=256, return_tensors="pt")
xlmr_enc_val = xlmr_tokenizer(val_summaries, truncation=True, padding=True, max_length=256, return_tensors="pt")
xlmr_enc_test = xlmr_tokenizer(test_summaries, truncation=True, padding=True, max_length=256, return_tensors="pt")

xlmr_ds_train = PriceDataset(xlmr_enc_train, y_train_log_t)
xlmr_ds_val = PriceDataset(xlmr_enc_val, y_val_log_t)

print("Loading XLM-R model (num_labels=1 → regression)...")
xlmr_model = AutoModelForSequenceClassification.from_pretrained(
    XLMR_NAME, num_labels=1, problem_type="regression"
)

xlmr_args = TrainingArguments(
    output_dir=str(XLMR_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=500,
    seed=SEED,
    report_to="none",
)

xlmr_trainer = Trainer(
    model=xlmr_model,
    args=xlmr_args,
    train_dataset=xlmr_ds_train,
    eval_dataset=xlmr_ds_val,
)

print("Training XLM-R...")
xlmr_trainer.train()
xlmr_trainer.save_model(str(XLMR_DIR))
xlmr_tokenizer.save_pretrained(str(XLMR_DIR))
print(f"Saved: {XLMR_DIR}")

# Evaluate
print("Evaluating XLM-R on test set...")
xlmr_model.eval()
xlmr_model.to(DEVICE)
pred_3_log = []
for i in range(0, len(test_items), batch_size):
    batch_enc = {k: v[i:i+batch_size].to(DEVICE) for k, v in xlmr_enc_test.items()}
    with torch.no_grad():
        out = xlmr_model(**batch_enc)
        pred_3_log.extend(out.logits.cpu().numpy().flatten())

pred_3 = np.expm1(np.array(pred_3_log))
pred_3 = np.clip(pred_3, 0, None)
evaluate_batch(test_prices, pred_3, "Model 3: XLM-R fine-tune")

del xlmr_model, xlmr_trainer
torch.cuda.empty_cache()


# ============================================================
# PHASE 7: SUMMARY
# ============================================================


In [ ]:
# Phase 7: Summary comparison

print("\n\n" + "=" * 70)
print("FINAL RESULTS — Day 4 Deep Learning + Day 3 Baseline")
print("=" * 70)

# Add Day 3 baseline for reference
ALL_RESULTS["Day 3 Baseline (Blended)"] = {
    "rmsle": 0.5164, "mae": 109_725, "mape": 44.0, "r2": 47.1
}

df = pd.DataFrame(ALL_RESULTS).T
df = df.sort_values("rmsle")
df.index.name = "Model"
print(df.to_string(
    formatters={
        "rmsle": "{:.4f}".format,
        "mae": "{:,.0f}".format,
        "mape": "{:.1f}%".format,
        "r2": "{:.1f}%".format,
    }
))

# Save results
results_path = DAY4 / "day4_results.json"
with open(results_path, "w") as f:
    json.dump(ALL_RESULTS, f, indent=2, ensure_ascii=False)
print(f"\nResults saved: {results_path}")

best_name = df.index[0]
best_rmsle = df.iloc[0]["rmsle"]
print(f"\nBest model: {best_name} (RMSLE={best_rmsle:.4f})")
print(f"Day 3 baseline: RMSLE=0.5164")
print(f"Improvement: {(0.5164 - best_rmsle) / 0.5164 * 100:.1f}%")
